# 06 — Hyperparameter Tuning
## Paddy Yield Predictor

We use `RandomizedSearchCV` to find the best settings for Random Forest.
RandomizedSearchCV is faster than GridSearchCV — it tries random combinations instead of every possible one.
After tuning, we save the best model to disk.

In [3]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.logger import get_logger
from src.data_loader import load_data, clean_data, split_features_target
from src.model_utils import build_preprocessor, evaluate_model, save_model

log = get_logger('06_hyperparameter_tuning')
log.info('Starting Hyperparameter Tuning notebook')

2026-08-18 15:12:10 | INFO     | 06_hyperparameter_tuning | Starting Hyperparameter Tuning notebook


In [4]:
from sklearn.model_selection import train_test_split

try:
    df = load_data(PROJECT_ROOT / 'paddydataset.csv')
    df = clean_data(df)
    X, y = split_features_target(df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    log.info('Data ready for tuning')
except Exception as e:
    log.error(f'Data preparation failed: {e}')
    raise

2026-08-18 15:12:10 | INFO     | src.data_loader | Dataset loaded — shape: (2789, 45)
2026-08-18 15:12:10 | INFO     | src.data_loader | Cleaning done — removed 451 duplicate/empty rows. Final shape: (2338, 45)
2026-08-18 15:12:10 | INFO     | src.data_loader | Features shape: (2338, 44) | Target shape: (2338,)
2026-08-18 15:12:10 | INFO     | 06_hyperparameter_tuning | Data ready for tuning


### Set up the parameter search space
These are the hyperparameters we want to try different values for.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

preprocessor = build_preprocessor(X)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# Parameters to try
# 'model__' prefix is needed because the estimator is inside a pipeline
param_grid = {
    'model__n_estimators'     : [200, 300, 500],
    'model__max_depth'        : [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf' : [1, 2, 4],
    'model__max_features'     : ['sqrt', 'log2', 1.0]
}

2026-08-18 15:12:10 | INFO     | src.model_utils | Numeric features: 36 | Categorical features: 8


In [6]:
# Run the search
# n_iter=15 means we try 15 random combinations (increase for better results)
# cv=3 means 3-fold cross validation
try:
    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_grid,
        n_iter=15,
        cv=3,
        scoring='neg_root_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train)

    print('Best parameters found:')
    print(search.best_params_)
    print('\nBest CV RMSE:', round(-search.best_score_, 2))

    log.info(f'Best params: {search.best_params_}')
    log.info(f'Best CV RMSE: {-search.best_score_:.2f}')

except Exception as e:
    log.error(f'RandomizedSearchCV failed: {e}')
    raise

Fitting 3 folds for each of 15 candidates, totalling 45 fits


2026-08-18 15:12:29 | INFO     | 06_hyperparameter_tuning | Best params: {'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 20}
2026-08-18 15:12:29 | INFO     | 06_hyperparameter_tuning | Best CV RMSE: 866.05


Best parameters found:
{'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 20}

Best CV RMSE: 866.05


In [7]:
# Evaluate the best model on the held-out test set
try:
    best_model = search.best_estimator_
    metrics = evaluate_model(best_model, X_test, y_test)

    print('Test set results:')
    print(f'  MAE       : {metrics["MAE"]} Kg')
    print(f'  RMSE      : {metrics["RMSE"]} Kg')
    print(f'  R² Score  : {metrics["R2 Score"]}')

except Exception as e:
    log.error(f'Evaluation failed: {e}')
    raise

2026-08-18 15:12:29 | INFO     | src.model_utils | Evaluation — MAE: 673.84 | RMSE: 934.16 | R²: 0.98981


Test set results:
  MAE       : 673.84 Kg
  RMSE      : 934.16 Kg
  R² Score  : 0.98981


In [8]:
# Save the best model
try:
    model_path = PROJECT_ROOT / 'models' / 'paddy_yield_predictor.pkl'
    save_model(best_model, model_path)
    print(f'Model saved to: {model_path}')
except Exception as e:
    log.error(f'Model save failed: {e}')
    raise

2026-08-18 15:12:30 | INFO     | src.model_utils | Model saved to: d:\TRUPTIMAYEE KHUNTIA\PADDY YIELD PREDICTOR\paddy_project\models\paddy_yield_predictor.pkl


Model saved to: d:\TRUPTIMAYEE KHUNTIA\PADDY YIELD PREDICTOR\paddy_project\models\paddy_yield_predictor.pkl
